In [1]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib

In [2]:
out_dir = "/p/scratch/hclimrep/lehner3/postprocessing/t2m"
scores_files = glob.glob(f"{out_dir}/*/metric_files/score_card/scores.csv")
scores_all = pd.concat([pd.read_csv(score_fname, index_col=0) for score_fname in scores_files], axis=0)

In [3]:
scores_all.head(3)

,eval_type,model,time,varname,score_name,value
0,temporal,test model,year,t2m,bias,0.027081
1,temporal,test model,MAM,t2m,bias,0.038328
2,temporal,test model,JJA,t2m,bias,0.360259


In [4]:
scores_tmp = scores_all.set_index(["eval_type", "model", "varname", "score_name"])
scores_tmp = scores_tmp.pivot(columns="time", values="value")[["year", "DJF", "MAM", "JJA", "SON"]].rename({"year": "YEAR"}, axis=1)
scores_reshaped = scores_tmp.reset_index()
scores_reshaped.head(3)

time,eval_type,model,varname,score_name,YEAR,DJF,MAM,JJA,SON
0,spatial,Harris WGAN,t2m,bias,0.037289,-0.110321,0.018394,0.160578,0.077733
1,spatial,Harris WGAN,t2m,rmse,1.017510,1.000537,1.005771,1.065024,0.998129
2,spatial,test model,t2m,bias,0.047289,-0.410321,0.028394,0.260578,0.047733


In [5]:
scores_reindexed = scores_reshaped.set_index(["eval_type", "varname", "score_name"])
scores_reindexed.head(3)

time                                model      YEAR       DJF       MAM  \
eval_type varname score_name                                              
spatial   t2m     bias        Harris WGAN  0.037289 -0.110321  0.018394   
                  rmse        Harris WGAN  1.017510  1.000537  1.005771   
                  bias         test model  0.047289 -0.410321  0.028394   

time                               JJA       SON  
eval_type varname score_name                      
spatial   t2m     bias        0.160578  0.077733  
                  rmse        1.065024  0.998129  
                  bias        0.260578  0.047733

In [6]:
for eval_group in scores_reindexed.groupby(scores_reindexed.index).groups:
    scores_iter = scores_reindexed.loc[eval_group].reset_index(drop=True)
    
    models = scores_iter["model"].values.tolist()
    time_aggs = scores_iter.columns[1:].values
    scores_values = scores_iter.drop("model", axis=1).values
    
    print(scores_iter)
    print(scores_values)
    print(models)
    print(time_aggs)
    break

time        model      YEAR       DJF       MAM       JJA       SON
0     Harris WGAN  0.037289 -0.110321  0.018394  0.160578  0.077733
1      test model  0.047289 -0.410321  0.028394  0.260578  0.047733
[[ 0.03728926 -0.11032114  0.01839389  0.16057792  0.07773331]
 [ 0.04728926 -0.41032114  0.02839389  0.26057792  0.04773331]]
['Harris WGAN', 'test model']
['YEAR' 'DJF' 'MAM' 'JJA' 'SON']


/tmp/ipykernel_2629352/769662253.py:2: PerformanceWarning: indexing past lexsort depth may impact performance.
  scores_iter = scores_reindexed.loc[eval_group].reset_index(drop=True)


In [165]:
units = {
    "bias": "K",
    "grad_amplitude": "1",
    "me_std": "K",
    "ralsd": "1",
    "rmse": "K",
}
vmins = {
    "bias": -0.5,
    "grad_amplitude": 0.9,
    "me_std": 0,
    "ralsd": 0,
    "rmse": 0,
}
vmaxs = {
    "bias": 0.5,
    "grad_amplitude": 1.1,
    "me_std": 1,
    "ralsd": 3,
    "rmse": 1.6,
}
varnames = {
    "t2m": "2m Temperature",
    "ws100m": "100m Wind Speed",
    "glob_rad": "Global Radiance",
}

In [166]:
for score in scores_reshaped.groupby("score_name").groups:
    print(score)
    vmin = vmins[score]
    vmax = vmaxs[score]
    score_unit = units[score]
    
    scores_iter = scores_all.query(f"score_name == '{score}'").drop(["score_name"], axis=1)
    g = sns.FacetGrid(scores_iter, col="varname", row="eval_type", height=3.5, aspect=2)
    
    # Create heatmap in each facet
    def heatmap_plot(data, color, **kws):
        pivot = data.pivot(index="model", columns="time", values="value")
        sns.heatmap(pivot, vmin=vmin, vmax=vmax, cbar=False, ax=plt.gca(), cmap="coolwarm",
                   square=True, linewidth=5, annot=True, fmt=".2f")
    
    g.map_dataframe(heatmap_plot)
    
    # Add a shared colorbar
    # Get the last axis to position colorbar correctly
    cbar_ax = g.fig.add_axes([0.11, 0., 0.85, 0.04])  # [left, bottom, width, height]
    
    # Draw colorbar separately using a dummy image
    import matplotlib as mpl
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=norm)
    sm.set_array([])  # dummy array for colorbar
    g.fig.colorbar(sm, cax=cbar_ax, label=f"{score.upper()} [{score_unit}]", orientation="horizontal")
    
    for gax in g.axes[-1]:
        gax.xaxis.tick_bottom()
        gax.set_xlabel("")
        
    for gax_ in g.axes:
        for gax in gax_:
            gax.set_ylabel("")
            gax.set_yticklabels(gax.get_yticklabels(), rotation=0)
            gax.set_title(f"{varnames[gax.get_title().split(' ')[-1]]}\n{gax.get_title().split(' ')[2]} {score}")
            for idx in range(len(models)):
                boundary = matplotlib.patches.Rectangle([0.03, idx+0.03], 4.94, 0.94, linewidth=2, color="darkgray", fc="none")
                gax.add_patch(boundary)
                for lineidx in range(1, 5):
                    linesbetween = matplotlib.patches.Rectangle([lineidx-0.017, idx+0.03], 0.035, 0.94, linewidth=2, color="darkgray")
                    gax.add_patch(linesbetween)
        
    plt.savefig(f"scorecard_{score}.png", bbox_inches="tight")

bias
grad_amplitude
me_std
ralsd
rmse


![](scorecard_bias.png)
![](scorecard_rmse.png)
![](scorecard_grad_amplitude.png)
![](scorecard_me_std.png)
![](scorecard_ralsd.png)